<a href="https://colab.research.google.com/github/Shahd-Saleem/AI-Powered-Exam-Proctoring-Camera/blob/main/cheating_camera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ultralytics

import torch
from ultralytics import YOLO
import cv2
import zipfile

In [ ]:
with zipfile.ZipFile("/content/drive/MyDrive/senior_project/head_movement_dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/head_dataset")

with zipfile.ZipFile("/content/drive/MyDrive/senior_project/phone_detection_dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/phone_dataset")

print("Datasets extracted")

Datasets extracted


In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [ ]:
#head_model = YOLO("yolov8n.pt")

#head_model.train(
 #   data="/content/head_dataset/data.yaml",
  #  epochs=30,
   # imgsz=640,
    #batch=8,
    #device=device,
    #project="/content/drive/MyDrive/senior_project",
    #name="head_model_3"
#)

head_model = YOLO("/content/drive/MyDrive/senior_project/head_model_3/weights/best.pt")

metrics = head_model.val(
    data="/content/head_dataset/data.yaml",
    imgsz=640
)

precision = metrics.box.mp
recall = metrics.box.mr
accuracy = (2 * precision * recall) / (precision + recall + 1e-6)

print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("Precision:", precision)
print("Recall:", recall)
print("Accuracy:", accuracy)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1132.2±290.5 MB/s, size: 38.2 KB)
val: Scanning /content/head_dataset/valid/labels.cache... 150 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 150/150 41.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.0s/it 10.1s
                   all        150        234      0.943      0.956       0.98      0.853
              cheating         91        108      0.963       0.96      0.984      0.857
                normal         76        126      0.923      0.952      0.977       0.85
Speed: 1.0ms preprocess, 59.2ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /content/runs/detect/val2
mAP50-95: 0.853441040146075
mAP50: 0.9803249064746392
Precision: 0.943021778235803
Recall: 0.9

In [ ]:
#phone_model = YOLO("yolov8n.pt")

#phone_model.train(
 #   data="/content/phone_dataset/data.yaml",
  #  epochs=30,
   # imgsz=640,
    #batch=8,
    #device=device,
    #project="/content/drive/MyDrive/senior_project",
    #name="phone_model_2"
#)

#metrics = phone_model.val(
 #   data="/content/phone_dataset/data.yaml",
  #  imgsz=640
#)

#precision = metrics.box.mp
#recall = metrics.box.mr
#accuracy = (2 * precision * recall) / (precision + recall + 1e-6)

#print("mAP50-95:", metrics.box.map)
#print("mAP50:", metrics.box.map50)
#print("Precision:", precision)
#print("Recall:", recall)
#print("Accuracy:", accuracy)

#phone_model = YOLO("/content/drive/MyDrive/senior_project/phone_model_2/weights/last.pt")
#phone_model.train(resume=True)

#metrics = phone_model.val(
#    data="/content/phone_dataset/data.yaml",
#    imgsz=640
#)

precision = metrics.box.mp
recall = metrics.box.mr
accuracy = (2 * precision * recall) / (precision + recall + 1e-6)

print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("Precision:", precision)
print("Recall:", recall)
print("Accuracy:", accuracy)

mAP50-95: 0.8515106463245748
mAP50: 0.9814251239896868
Precision: 0.9432157642121315
Recall: 0.957821874630896
Accuracy: 0.9504622083945714


In [ ]:
from ultralytics import YOLO

head_model = YOLO("/content/drive/MyDrive/senior_project/head_model_3/weights/best.pt")
phone_model = YOLO("/content/drive/MyDrive/senior_project/phone_model_2/weights/best.pt")
person_counter = YOLO('yolov8n.pt')

print("Models loaded")

Models loaded


In [ ]:
'''
import cv2
import numpy as np
import traceback
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
from ultralytics import YOLO

#CHEATING LOGIC
def apply_yolo_cheating_logic(img, consecutive_cheating_frames):
    phone_detected_alarm = False
    is_looking_away = False

    phone_results = phone_model(img, verbose=False)
    head_results = head_model(img, verbose=False, agnostic_nms=True)

#phone detection:
    for r in phone_results:
        for box in r.boxes:
            conf = float(box.conf[0])
            if conf >= 0.40:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                phone_detected_alarm = True
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(img, f"Phone! {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

#head detection:
    valid_heads_for_cheating =[]
    total_people_in_frame = 0

    for r in head_results:
        for box in r.boxes:
            conf = float(box.conf[0])
            if conf >= 0.35:
                total_people_in_frame += 1

            # Must still be > 50% confident to actually judge if it's 'cheating' or 'normal'
            if conf >= 0.50:
                valid_heads_for_cheating.append(box)

    for box in valid_heads_for_cheating:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        class_name = head_model.names[cls_id].lower()

        if class_name == 'cheating':
            if total_people_in_frame >= 2:
                is_looking_away = True
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 165, 255), 2) # Orange Box
                cv2.putText(img, f"Looking at someone! {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 165, 255), 2)
            else:
                # Alone
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 0), 2) # Cyan Box
                cv2.putText(img, f"Looking away (Alone) {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)
        else:
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)   # Green Box
            cv2.putText(img, f"Normal {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    #timer:
    if is_looking_away:
        consecutive_cheating_frames += 1
    else:
        consecutive_cheating_frames = max(0, consecutive_cheating_frames - 2)

    #DEBUGGING:
    cv2.putText(img, f"People in Frame: {total_people_in_frame}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(img, f"Cheating Timer: {consecutive_cheating_frames}/8", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    # cheating detected:
    cheating_alarm_active = False

    if phone_detected_alarm or consecutive_cheating_frames >= 8:
        cheating_alarm_active = True

    if cheating_alarm_active:
        cv2.putText(img, "WARNING: CHEATING DETECTED!", (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        cv2.rectangle(img, (0,0), (img.shape[1], img.shape[0]), (0, 0, 255), 10)

    return img, consecutive_cheating_frames

# COLAB JAVASCRIPT WEBCAM:
def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

def bbox_to_bytes(bbox_array):
    _, bbox_buffer = cv2.imencode('.jpg', bbox_array)
    bbox_bytes = b64encode(bbox_buffer).decode('utf-8')
    return bbox_bytes

def video_stream():
  js = Javascript('''
    var video; var div = null; var stream; var captureCanvas; var imgElement; var labelElement;
    var pendingResolve = null; var shutdown = false;
    function removeDom() {
       if(stream){stream.getVideoTracks()[0].stop();}
       if(video){video.remove();}
       if(div){div.remove();}
       video = null; div = null; stream = null; imgElement = null; captureCanvas = null; labelElement = null;
    }
    function onAnimationFrame() {
      if (!shutdown) { window.requestAnimationFrame(onAnimationFrame); }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve; pendingResolve = null; lp(result);
      }
    }
    async function createDom() {
      if (div !== null) { return stream; }
      div = document.createElement('div');
      div.style.border = '2px solid black'; div.style.padding = '3px'; div.style.width = '100%'; div.style.maxWidth = '600px';
      document.body.appendChild(div);
      const modelOut = document.createElement('div');
      modelOut.innerHTML = "<span>Status: WebCam Running (Click video to stop)</span>";
      labelElement = document.createElement('span'); div.appendChild(modelOut);
      video = document.createElement('video'); video.style.display = 'block'; video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', ''); video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia({video: { facingMode: "user"}});
      div.appendChild(video);
      imgElement = document.createElement('img'); imgElement.style.position = 'absolute'; imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; }; div.appendChild(imgElement);
      video.srcObject = stream; await video.play();
      captureCanvas = document.createElement('canvas'); captureCanvas.width = 640; captureCanvas.height = 480;
      window.requestAnimationFrame(onAnimationFrame); return stream;
    }
    async function stream_frame(label, imgData) {
      if (shutdown) { removeDom(); shutdown = false; return ''; }
      var preCreate = Date.now(); stream = await createDom();
      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px"; imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px"; imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData;
      }
      var result = await new Promise(function(resolve, reject) { pendingResolve = resolve; });
      shutdown = false; return {'img': result};
    }
    ''')
  display(js)

def video_frame(label, bbox):
  data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
  return data

# LIVE DETECTION:
print("Starting webcam... Please allow camera access in your browser.")
video_stream()
label_html = 'Running Cheating Detection...'
bbox = ''

consecutive_cheating_frames = 0

while True:
    try:
        js_reply = video_frame(label_html, bbox)

        if not js_reply or not js_reply.get("img"):
            print("Video stopped by user.")
            break

        img = js_to_image(js_reply["img"])
        processed_img, consecutive_cheating_frames = apply_yolo_cheating_logic(img, consecutive_cheating_frames)
        bbox = "data:image/jpeg;base64," + bbox_to_bytes(processed_img)

    except Exception as e:
        print("\n❌ CRASH DETECTED!")
        print(f"Error message: {e}")
        traceback.print_exc()
        print("Stopping stream...")
        break

print("Webcam successfully closed.")
'''

Starting webcam... Please allow camera access in your browser.


<IPython.core.display.Javascript object>


❌ CRASH DETECTED!
Error message: NotAllowedError: Permission denied
Stopping stream...
Webcam successfully closed.


Traceback (most recent call last):
  File "/tmp/ipykernel_9146/2985334431.py", line 185, in <cell line: 0>
    js_reply = video_frame(label_html, bbox)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_9146/2985334431.py", line 170, in video_frame
    data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/colab/output/_js.py", line 40, in eval_js
    return _message.read_reply_from_input(request_id, timeout_sec)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_message.py", line 103, in read_reply_from_input
    raise MessageError(reply['error'])
google.colab._message.MessageError: NotAllowedError: Permission denied


In [ ]:
!pip uninstall mediapipe -y
!pip install mediapipe==0.10.21

Found existing installation: mediapipe 0.10.21
Uninstalling mediapipe-0.10.21:
  Successfully uninstalled mediapipe-0.10.21
  Using cached mediapipe-0.10.21-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (9.7 kB)
Using cached mediapipe-0.10.21-cp312-cp312-manylinux_2_28_x86_64.whl (35.6 MB)


In [ ]:
import mediapipe as mp
from ultralytics import YOLO

# Load Phone Model
phone_model = YOLO("/content/drive/MyDrive/senior_project/phone_model_2/weights/last.pt")

# Load Person Counter
person_counter = YOLO('yolov8n.pt')

# Initialize MediaPipe Face Tracking:
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    max_num_faces=5
)

print("Models and MediaPipe successfully loaded!")

Models and MediaPipe successfully loaded!


In [ ]:
import cv2
import numpy as np
import traceback
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode

# CHEATING LOGIC (YOLO + MediaPipe):
# cheating logic code:
def apply_hybrid_cheating_logic(img, consecutive_cheating_frames):
    phone_detected_alarm = False
    is_looking_away = False

    img_h, img_w, _ = img.shape

    # DETECT PHONE (YOLO):
    phone_results = phone_model(img, verbose=False)
    for r in phone_results:
        for box in r.boxes:
            conf = float(box.conf[0])
            if conf >= 0.40:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                phone_detected_alarm = True
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                cv2.putText(img, f"Phone! {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

    # FIND WHERE PEOPLE ARE SITTING (YOLO):
    person_centers = []
    person_results = person_counter(img, classes=[0], verbose=False)

    for r in person_results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            # Find the center X coordinate of this person's body:
            center_x = (x1 + x2) // 2
            person_centers.append(center_x)

    total_people_in_frame = len(person_centers)

    # PROCESS HEAD (MediaPipe):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    face_results = face_mesh.process(img_rgb)

    if face_results.multi_face_landmarks:
        for face_landmarks in face_results.multi_face_landmarks:
            face_2d = []
            face_3d =[]

            for idx, lm in enumerate(face_landmarks.landmark):
                if idx in[33, 263, 1, 61, 291, 199]:
                    if idx == 1: # Nose tip
                        nose_2d = (int(lm.x * img_w), int(lm.y * img_h))
                    x, y = int(lm.x * img_w), int(lm.y * img_h)
                    face_2d.append([x, y])
                    face_3d.append([x, y, lm.z])

            face_2d = np.array(face_2d, dtype=np.float64)
            face_3d = np.array(face_3d, dtype=np.float64)

            focal_length = 1 * img_w
            cam_matrix = np.array([[focal_length, 0, img_h / 2],[0, focal_length, img_w / 2], [0, 0, 1]])
            dist_matrix = np.zeros((4, 1), dtype=np.float64)

            success, rot_vec, trans_vec = cv2.solvePnP(face_3d, face_2d, cam_matrix, dist_matrix)
            rmat, jac = cv2.Rodrigues(rot_vec)
            angles, mtxR, mtxQ, Qx, Qy, Qz = cv2.RQDecomp3x3(rmat)

            # Get Angles:
            y_angle = angles[1] * 360  # Yaw (Left/Right)
            x_angle = angles[0] * 360  # Pitch (Up/Down)

            nose_x = nose_2d[0]

            # CHECK SURROUNDINGS: Are there bodies to the left or right of THIS face?
            # We use 100 pixels as a buffer so it doesn't accidentally count the person's own body
            people_to_the_left = sum(1 for px in person_centers if px < nose_x - 100)
            people_to_the_right = sum(1 for px in person_centers if px > nose_x + 100)

            # Determine direction of head turn:
            looking_screen_left = y_angle < -15
            looking_screen_right = y_angle > 15
            looking_down = x_angle < -15

            is_cheating_turn = False
            status_text = "Forward"
            dot_color = (0, 255, 0) # Green

            # SPATIAL AWARENESS LOGIC:
            if looking_screen_left:
                if people_to_the_left > 0:
                    is_cheating_turn = True
                    status_text = "Looking at friend (Left)!"
                    dot_color = (0, 165, 255) # Orange
                else:
                    status_text = "Looking away (Safe)"
                    dot_color = (255, 255, 0) # Cyan

            elif looking_screen_right:
                if people_to_the_right > 0:
                    is_cheating_turn = True
                    status_text = "Looking at friend (Right)!"
                    dot_color = (0, 165, 255)
                else:
                    status_text = "Looking away (Safe)"
                    dot_color = (255, 255, 0)

            elif looking_down:
                status_text = "Looking down (Safe)"
                dot_color = (255, 255, 0)

            # Apply text based on status:
            if is_cheating_turn:
                is_looking_away = True

            cv2.putText(img, status_text, (nose_x-50, nose_2d[1]-20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, dot_color, 2)
            cv2.circle(img, nose_2d, 5, dot_color, -1)

    # TIMER:
    if is_looking_away:
        consecutive_cheating_frames += 1
    else:
        consecutive_cheating_frames = max(0, consecutive_cheating_frames - 2)

    # DEBUGGING & ALARMS
    cv2.putText(img, f"People in Frame: {total_people_in_frame}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    cv2.putText(img, f"Cheating Timer: {consecutive_cheating_frames}/8", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

    cheating_alarm_active = False
    if phone_detected_alarm or consecutive_cheating_frames >= 8:
        cheating_alarm_active = True

    if cheating_alarm_active:
        cv2.putText(img, "WARNING: CHEATING DETECTED!", (20, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
        cv2.rectangle(img, (0,0), (img.shape[1], img.shape[0]), (0, 0, 255), 10)

    return img, consecutive_cheating_frames

# COLAB JAVASCRIPT WEBCAM:
def js_to_image(js_reply):
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

def bbox_to_bytes(bbox_array):
    _, bbox_buffer = cv2.imencode('.jpg', bbox_array)
    bbox_bytes = b64encode(bbox_buffer).decode('utf-8')
    return bbox_bytes

def video_stream():
  js = Javascript('''
    var video; var div = null; var stream; var captureCanvas; var imgElement; var labelElement;
    var pendingResolve = null; var shutdown = false;
    function removeDom() {
       if(stream){stream.getVideoTracks()[0].stop();}
       if(video){video.remove();}
       if(div){div.remove();}
       video = null; div = null; stream = null; imgElement = null; captureCanvas = null; labelElement = null;
    }
    function onAnimationFrame() {
      if (!shutdown) { window.requestAnimationFrame(onAnimationFrame); }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve; pendingResolve = null; lp(result);
      }
    }
    async function createDom() {
      if (div !== null) { return stream; }
      div = document.createElement('div');
      div.style.border = '2px solid black'; div.style.padding = '3px'; div.style.width = '100%'; div.style.maxWidth = '600px';
      document.body.appendChild(div);
      const modelOut = document.createElement('div');
      modelOut.innerHTML = "<span>Status: WebCam Running (Click video to stop)</span>";
      labelElement = document.createElement('span'); div.appendChild(modelOut);
      video = document.createElement('video'); video.style.display = 'block'; video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', ''); video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia({video: { facingMode: "user"}});
      div.appendChild(video);
      imgElement = document.createElement('img'); imgElement.style.position = 'absolute'; imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; }; div.appendChild(imgElement);
      video.srcObject = stream; await video.play();
      captureCanvas = document.createElement('canvas'); captureCanvas.width = 640; captureCanvas.height = 480;
      window.requestAnimationFrame(onAnimationFrame); return stream;
    }
    async function stream_frame(label, imgData) {
      if (shutdown) { removeDom(); shutdown = false; return ''; }
      var preCreate = Date.now(); stream = await createDom();
      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px"; imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px"; imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData;
      }
      var result = await new Promise(function(resolve, reject) { pendingResolve = resolve; });
      shutdown = false; return {'img': result};
    }
    ''')
  display(js)

def video_frame(label, bbox):
  data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
  return data

# LIVE DETECTION LOOP:
print("Starting webcam... Please allow camera access in your browser.")
video_stream()
label_html = 'Running Cheating Detection...'
bbox = ''

consecutive_cheating_frames = 0

while True:
    try:
        js_reply = video_frame(label_html, bbox)

        if not js_reply or not js_reply.get("img"):
            print("Video stopped by user.")
            break

        img = js_to_image(js_reply["img"])

        # Now using our new Hybrid (MediaPipe + YOLO) function:
        processed_img, consecutive_cheating_frames = apply_hybrid_cheating_logic(img, consecutive_cheating_frames)

        bbox = "data:image/jpeg;base64," + bbox_to_bytes(processed_img)

    except Exception as e:
        print("\n❌ CRASH DETECTED!")
        print(f"Error message: {e}")
        traceback.print_exc()
        print("Stopping stream...")
        break

print("Webcam successfully closed.")

Starting webcam... Please allow camera access in your browser.


<IPython.core.display.Javascript object>

Video stopped by user.
Webcam successfully closed.
